# 📊 Exploratory Data Analysis (EDA)
### Multimodal Medical Report Generator (IU Chest X-Ray Dataset)

This notebook analyzes the processed IU Chest X-Ray dataset to understand the pairing between images and text reports, token lengths, text distribution, and image features after CLAHE enhancement.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import albumentations as A
import sys

# Ensure local src imports work
sys.path.append(os.path.abspath(".."))
from src.data.augmentations import get_transforms

%matplotlib inline

## 1. Load Dataset Splits
Let's load the generated train, validation, and test splits to check balance and formatting.

In [ ]:
splits_dir = "../data/splits"
train_df = pd.read_csv(os.path.join(splits_dir, "train.csv"))
val_df = pd.read_csv(os.path.join(splits_dir, "val.csv"))
test_df = pd.read_csv(os.path.join(splits_dir, "test.csv"))
metadata_df = pd.read_csv(os.path.join(splits_dir, "metadata.csv"))

print(f"Train samples: {len(train_df)}")
print(f"Val samples:   {len(val_df)}")
print(f"Test samples:  {len(test_df)}")
print(f"Total unique reports: {metadata_df['report_id'].nunique()}")
print(f"Total images: {len(metadata_df)}")

In [ ]:
# Display first few rows
metadata_df.head()

## 2. Analyze Report Text Characteristics
Understanding the distribution of findings and impression lengths is critical for setting the maximum sequence length (`max_length`) in the tokenizer and preventing truncation of key clinical information.

In [ ]:
# Compute character and word counts
metadata_df['findings_len'] = metadata_df['findings'].fillna("").apply(lambda x: len(x.split()))
metadata_df['impression_len'] = metadata_df['impression'].fillna("").apply(lambda x: len(x.split()))
metadata_df['total_report_len'] = metadata_df['text_report'].fillna("").apply(lambda x: len(x.split()))

print("--- Word Count Statistics ---")
print(metadata_df[['findings_len', 'impression_len', 'total_report_len']].describe())

# Plot word count distribution
plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.hist(metadata_df['findings_len'], bins=20, color='skyblue', edgecolor='black')
plt.title('Findings Word Count')
plt.xlabel('Words')
plt.ylabel('Frequency')

plt.subplot(1, 3, 2)
plt.hist(metadata_df['impression_len'], bins=20, color='salmon', edgecolor='black')
plt.title('Impression Word Count')
plt.xlabel('Words')

plt.subplot(1, 3, 3)
plt.hist(metadata_df['total_report_len'], bins=20, color='lightgreen', edgecolor='black')
plt.title('Total Report Word Count')
plt.xlabel('Words')

plt.tight_layout()
plt.show()

## 3. High-Frequency Terms
Let's see what the most common terms in the radiology findings are. This informs vocabulary expectations.

In [ ]:
from collections import Counter
import re

all_findings_text = " ".join(metadata_df['findings'].dropna().tolist()).lower()
words = re.findall(r'\w+', all_findings_text)
stopwords = {'the', 'and', 'are', 'is', 'of', 'in', 'with', 'no', 'or', 'for', 'to', 'at', 'on', 'any', 'that', 'there', 'be', 'this', 'an', 'as'}
filtered_words = [w for w in words if w not in stopwords]

word_counts = Counter(filtered_words)
common_words = pd.DataFrame(word_counts.most_common(20), columns=['Word', 'Count'])

plt.figure(figsize=(10, 5))
plt.bar(common_words['Word'], common_words['Count'], color='teal')
plt.title('Top 20 Most Frequent Clinical Terms (Stopwords Removed)')
plt.xticks(rotation=45, ha='right')
plt.ylabel('Count')
plt.show()

## 4. Visualize X-Rays and Preprocessing (CLAHE)
Chest X-rays have very narrow histogram dynamic ranges. Let's compare standard normalization against CLAHE (Contrast Limited Adaptive Histogram Equalization) to see how it enhances local structures.

In [ ]:
images_dir = "../data/raw/images"
sample_row = metadata_df.iloc[0]
sample_img_path = os.path.join(images_dir, sample_row['image_name'])

if os.path.exists(sample_img_path):
    img = Image.open(sample_img_path).convert("RGB")
    img_np = np.array(img)
    
    # Initialize transforms
    clahe_only = A.Compose([A.Resize(224, 224), A.CLAHE(clip_limit=2.0, p=1.0)])
    no_clahe = A.Compose([A.Resize(224, 224)])
    
    img_resized = no_clahe(image=img_np)['image']
    img_clahe = clahe_only(image=img_np)['image']
    
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(img_resized)
    plt.title('Original (Resized 224x224)')
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.imshow(img_clahe)
    plt.title('CLAHE Enhanced (224x224)')
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print(f"Report for this scan:\n{sample_row['text_report']}")
else:
    print("Sample image not found. Execute preprocessing script to download and parse files first.")